# Практическая работа 11-12. Кластерный анализ

Дисциплина: Интеллектуальные системы и технологии

**Вариант 10.** Наборы данных:
- из таблицы 1 (с известными метками): Haberman's Survival
- из таблицы 2 (с неизвестными метками): Human Activity Recognition Using Smartphones (файл `test.csv`)
- дополнительно: `mammoth.csv` для финальной визуализации

**Задача.** Применить иерархические и итерационные алгоритмы кластеризации, оценить оптимальное число кластеров на интервале от 2 до 10 с помощью внутренних индексов (силуэт, elbow, Davies-Bouldin, Calinski-Harabasz) и внешних индексов (ARI, NMI) там, где известны истинные метки. Выбрать лучший алгоритм, визуализировать результаты с помощью t-SNE, UMAP, TriMAP и PaCMAP, выполнить профилирование кластеров.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering, Birch
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score, normalized_mutual_info_score
)
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import cdist

from sklearn.manifold import TSNE
import umap
import trimap
import pacmap

np.random.seed(42)
RND = 42

## Часть 1. Haberman's Survival — датасет с известными метками классов

Датасет содержит данные о пациентах, перенёсших операцию по поводу рака молочной железы. Признаки: возраст пациента на момент операции, год операции (минус 1900), число обнаруженных подмышечных лимфоузлов. Истинная метка — выжил пациент более 5 лет (класс 1) или нет (класс 2).

В рамках практики метки рассматриваются как **неизвестные на этапе кластеризации** и используются только для финальной оценки качества внешними индексами (ARI, NMI).

In [ ]:
# Загрузка датасета Haberman
hab = pd.read_csv('haberman.csv', header=None,
                  names=['age', 'year', 'nodes', 'status'])
print(f"Размер: {hab.shape}")
print(f"\nРаспределение классов:\n{hab['status'].value_counts().to_string()}")
print(f"\nСтатистика по признакам:\n{hab.describe().round(2)}")

X_hab = hab[['age', 'year', 'nodes']].values
y_hab_true = hab['status'].values

scaler = StandardScaler()
X_hab_sc = scaler.fit_transform(X_hab)

### 1.1. Разведочный анализ

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

for ax, col in zip(axes[:3], ['age', 'year', 'nodes']):
    for cls, color in [(1, '#1f77b4'), (2, '#d62728')]:
        ax.hist(hab[hab['status'] == cls][col], bins=20, alpha=0.6,
                color=color, label=f'Класс {cls}')
    ax.set_title(col); ax.legend(fontsize=8)

# Корреляции между признаками
corr = hab[['age', 'year', 'nodes']].corr()
im = axes[3].imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
axes[3].set_xticks(range(3)); axes[3].set_yticks(range(3))
axes[3].set_xticklabels(corr.columns); axes[3].set_yticklabels(corr.columns)
for i in range(3):
    for j in range(3):
        axes[3].text(j, i, f'{corr.values[i, j]:.2f}',
                     ha='center', va='center', color='black')
axes[3].set_title('Корреляции')

plt.tight_layout()
plt.show()

### 1.2. Иерархическая кластеризация — дендрограмма

Используем алгоритм агломеративной иерархической кластеризации с методом Уорда (минимизация внутрикластерной дисперсии). Дендрограмма позволяет увидеть структуру слияния кластеров и подсказать «естественное» число кластеров по самому большому скачку расстояния.

In [ ]:
Z = linkage(X_hab_sc, method='ward')

fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(Z, truncate_mode='lastp', p=30, ax=ax,
           leaf_rotation=90, show_contracted=True)
ax.set_title('Дендрограмма (Ward, Haberman)')
ax.set_xlabel('Объекты (сжатое представление)')
ax.set_ylabel('Расстояние слияния')
plt.tight_layout()
plt.show()

### 1.3. Поиск оптимального числа кластеров

Для каждого алгоритма (KMeans, агломеративный Ward, GaussianMixture, Spectral) и каждого k от 2 до 10 считаем:
- **Silhouette** — насколько кластеры компактны и отделены друг от друга (от -1 до 1, выше — лучше)
- **Davies-Bouldin** — среднее отношение внутрикластерной дисперсии к межкластерной (ниже — лучше)
- **Calinski-Harabasz** — отношение межкластерной дисперсии к внутрикластерной (выше — лучше)
- **ARI** и **NMI** — внешние индексы относительно истинных меток (выше — лучше). Они доступны только потому, что для Haberman метки фактически известны.
- **Inertia** для метода локтя — сумма квадратов расстояний до ближайшего центроида

In [ ]:
def fit_predict(name, k, X):
    if name == 'KMeans':
        m = KMeans(n_clusters=k, n_init=10, random_state=RND)
        labels = m.fit_predict(X)
        centers = m.cluster_centers_
        inertia = m.inertia_
    elif name == 'Ward':
        m = AgglomerativeClustering(n_clusters=k, linkage='ward')
        labels = m.fit_predict(X)
        centers = np.array([X[labels == i].mean(axis=0) for i in range(k)])
        inertia = sum(((X[labels == i] - centers[i])**2).sum() for i in range(k))
    elif name == 'GMM':
        m = GaussianMixture(n_components=k, random_state=RND, covariance_type='full')
        labels = m.fit_predict(X)
        centers = m.means_
        inertia = sum(((X[labels == i] - centers[i])**2).sum() for i in range(k))
    elif name == 'Spectral':
        m = SpectralClustering(n_clusters=k, random_state=RND,
                               affinity='nearest_neighbors', n_neighbors=15,
                               assign_labels='kmeans')
        labels = m.fit_predict(X)
        centers = np.array([X[labels == i].mean(axis=0) for i in range(k)])
        inertia = sum(((X[labels == i] - centers[i])**2).sum() for i in range(k))
    return labels, centers, inertia

def evaluate_all(X, y_true, algos, k_range):
    rows = []
    for name in algos:
        for k in k_range:
            labels, _, inertia = fit_predict(name, k, X)
            if len(np.unique(labels)) < 2:
                continue
            rows.append({
                'Алгоритм': name, 'k': k,
                'Silhouette': silhouette_score(X, labels),
                'Davies-Bouldin': davies_bouldin_score(X, labels),
                'Calinski-Harabasz': calinski_harabasz_score(X, labels),
                'Inertia': inertia,
                'ARI': adjusted_rand_score(y_true, labels) if y_true is not None else np.nan,
                'NMI': normalized_mutual_info_score(y_true, labels) if y_true is not None else np.nan,
            })
    return pd.DataFrame(rows)

k_range = list(range(2, 11))
algos = ['KMeans', 'Ward', 'GMM', 'Spectral']

scores_hab = evaluate_all(X_hab_sc, y_hab_true, algos, k_range)
print("Метрики качества кластеризации на Haberman:")
print(scores_hab.round(4).to_string(index=False))

In [ ]:
# Графики метрик в зависимости от k
metrics_to_plot = ['Silhouette', 'Davies-Bouldin', 'Calinski-Harabasz', 'Inertia', 'ARI', 'NMI']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, metric in zip(axes.flat, metrics_to_plot):
    for name in algos:
        sub = scores_hab[scores_hab['Алгоритм'] == name]
        ax.plot(sub['k'], sub[metric], marker='o', label=name)
    ax.set_title(metric)
    ax.set_xlabel('k')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    if metric == 'Inertia':
        ax.set_title('Inertia (метод локтя)')

plt.suptitle('Метрики кластеризации на Haberman', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Лучшие комбинации (алгоритм, k) по разным метрикам
print("Лучшие комбинации алгоритм + k по каждой метрике:")
for metric, best_func in [
    ('Silhouette', 'idxmax'),
    ('Davies-Bouldin', 'idxmin'),
    ('Calinski-Harabasz', 'idxmax'),
    ('ARI', 'idxmax'),
    ('NMI', 'idxmax'),
]:
    idx = getattr(scores_hab[metric], best_func)()
    row = scores_hab.loc[idx]
    print(f"  {metric:20s}: {row['Алгоритм']:10s} k={int(row['k'])}, "
          f"значение = {row[metric]:.4f}")

### 1.4. Выбор лучшего алгоритма и визуализация

Поскольку для Haberman истинное число классов равно 2, наиболее объективным критерием является ARI (внешний индекс) при k=2. Внутренние индексы могут указывать на другое оптимальное k, поскольку признаки слабо разделяют истинные классы — это известная особенность датасета.

In [ ]:
# Берём лучший по ARI алгоритм при k=2
best_at_k2 = scores_hab[scores_hab['k'] == 2].sort_values('ARI', ascending=False).iloc[0]
best_algo_hab = best_at_k2['Алгоритм']
print(f"Лучший алгоритм для Haberman при k=2 по ARI: {best_algo_hab}")
print(f"  ARI = {best_at_k2['ARI']:.4f}")
print(f"  NMI = {best_at_k2['NMI']:.4f}")
print(f"  Silhouette = {best_at_k2['Silhouette']:.4f}")

labels_hab, centers_hab, _ = fit_predict(best_algo_hab, 2, X_hab_sc)

# Понижение размерности для визуализации
def reduce_all(X, n_neighbors=15, perplexity=30):
    X32 = X.astype(np.float32)
    return {
        't-SNE':  TSNE(n_components=2, perplexity=perplexity, random_state=RND).fit_transform(X),
        'UMAP':   umap.UMAP(n_neighbors=n_neighbors, min_dist=0.1, random_state=RND).fit_transform(X),
        'TriMAP': trimap.TRIMAP(n_dims=2, verbose=False).fit_transform(X32),
        'PaCMAP': pacmap.PaCMAP(n_components=2, n_neighbors=n_neighbors, random_state=RND).fit_transform(X),
    }

print("Применение методов снижения размерности к Haberman...")
emb_hab = reduce_all(X_hab_sc)
print("Готово")

In [ ]:
def plot_clusters(X_emb, labels, title, ax, centers_emb=None):
    n = len(np.unique(labels))
    cmap = plt.get_cmap('tab10' if n <= 10 else 'tab20')
    for i, cls in enumerate(np.unique(labels)):
        mask = labels == cls
        ax.scatter(X_emb[mask, 0], X_emb[mask, 1],
                   c=[cmap(i)], s=25, alpha=0.6,
                   label=f'Кл {cls}', edgecolors='white', linewidth=0.2)
    if centers_emb is not None:
        ax.scatter(centers_emb[:, 0], centers_emb[:, 1],
                   c='black', marker='X', s=200,
                   edgecolors='white', linewidth=1.5, label='Центроиды', zorder=10)
    ax.set_title(title, fontsize=10)
    ax.legend(loc='best', fontsize=7)
    ax.set_xticks([]); ax.set_yticks([])

# Чтобы показать центроиды на embedding-ах, проецируем их через ближайших соседей в пространстве признаков
def project_centers(X_orig, X_emb, centers_orig):
    proj = []
    for c in centers_orig:
        d = np.linalg.norm(X_orig - c, axis=1)
        idx = np.argmin(d)
        proj.append(X_emb[idx])
    return np.array(proj)

# Сравнение: метки кластеризации vs истинные метки
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for j, method in enumerate(['t-SNE', 'UMAP', 'TriMAP', 'PaCMAP']):
    centers_proj = project_centers(X_hab_sc, emb_hab[method], centers_hab)
    plot_clusters(emb_hab[method], labels_hab,
                  f'{method}: метки {best_algo_hab}', axes[0, j], centers_proj)
    plot_clusters(emb_hab[method], y_hab_true,
                  f'{method}: истинные метки', axes[1, j])

plt.suptitle(f'Haberman: кластеризация {best_algo_hab} (k=2) vs истинные классы', fontsize=13)
plt.tight_layout()
plt.show()

### 1.5. Профилирование кластеров Haberman

In [ ]:
profile_hab = pd.DataFrame(X_hab, columns=['age', 'year', 'nodes'])
profile_hab['cluster'] = labels_hab
profile_hab['true_status'] = y_hab_true

print("Средние значения признаков по кластерам:")
print(profile_hab.groupby('cluster')[['age', 'year', 'nodes']].mean().round(2))

print("\nРазмеры кластеров:")
print(profile_hab['cluster'].value_counts().sort_index().to_string())

print("\nКросс-таблица: кластер x истинный класс:")
print(pd.crosstab(profile_hab['cluster'], profile_hab['true_status'],
                  rownames=['cluster'], colnames=['true_class']))

## Часть 2. HAR Smartphones (test.csv) — датасет с неизвестными метками

Датасет Human Activity Recognition Using Smartphones содержит признаки, извлечённые из показаний акселерометра и гироскопа смартфонов, носимых испытуемыми. В файле `test.csv` 561 признак и 2947 наблюдений. По условию варианта метки рассматриваются как неизвестные, но в данных они присутствуют (`Activity` — 6 классов: WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS, SITTING, STANDING, LAYING) и используются для финальной оценки внешними индексами.

Из-за высокой размерности (561) применяем PCA до 50 компонент перед кластеризацией — это ускоряет вычисления и снижает шум, не теряя существенной информации (PCA-50 обычно сохраняет более 90% дисперсии для HAR).

In [ ]:
har = pd.read_csv('test.csv')
print(f"Размер: {har.shape}")
print(f"\nРаспределение классов Activity:\n{har['Activity'].value_counts().to_string()}")

feature_cols = [c for c in har.columns if c not in ('subject', 'Activity')]
X_har_raw = har[feature_cols].values
y_har_str = har['Activity'].values

# Числовые метки для расчёта внешних индексов
activity_to_int = {a: i for i, a in enumerate(sorted(np.unique(y_har_str)))}
y_har_true = np.array([activity_to_int[a] for a in y_har_str])
print(f"\nКодирование меток: {activity_to_int}")

scaler_har = StandardScaler()
X_har_sc = scaler_har.fit_transform(X_har_raw)

pca = PCA(n_components=50, random_state=RND)
X_har = pca.fit_transform(X_har_sc)
print(f"PCA: {pca.n_components_} компонент сохраняют "
      f"{pca.explained_variance_ratio_.sum() * 100:.1f}% дисперсии")

### 2.1. Дендрограмма HAR

In [ ]:
Z_har = linkage(X_har, method='ward')
fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(Z_har, truncate_mode='lastp', p=40, ax=ax,
           leaf_rotation=90, show_contracted=True)
ax.set_title('Дендрограмма (Ward, HAR Smartphones)')
ax.set_xlabel('Объекты (сжатое представление)')
ax.set_ylabel('Расстояние слияния')
plt.tight_layout()
plt.show()

### 2.2. Поиск оптимального числа кластеров на HAR

In [ ]:
# Spectral на 2947 точках с n_neighbors=15 будет слишком медленным,
# поэтому ограничимся тремя алгоритмами
algos_har = ['KMeans', 'Ward', 'GMM']

scores_har = evaluate_all(X_har, y_har_true, algos_har, k_range)
print("Метрики качества кластеризации на HAR Smartphones:")
print(scores_har.round(4).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, metric in zip(axes.flat, metrics_to_plot):
    for name in algos_har:
        sub = scores_har[scores_har['Алгоритм'] == name]
        ax.plot(sub['k'], sub[metric], marker='o', label=name)
    ax.set_title(metric)
    ax.set_xlabel('k')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    if metric == 'Inertia':
        ax.set_title('Inertia (метод локтя)')

# Вертикальная линия — истинное число классов
for ax in axes.flat:
    ax.axvline(6, color='red', linestyle='--', alpha=0.5, label='истинное k=6')

plt.suptitle('Метрики кластеризации на HAR Smartphones', fontsize=13)
plt.tight_layout()
plt.show()

print("\nЛучшие комбинации алгоритм + k по каждой метрике:")
for metric, best_func in [
    ('Silhouette', 'idxmax'),
    ('Davies-Bouldin', 'idxmin'),
    ('Calinski-Harabasz', 'idxmax'),
    ('ARI', 'idxmax'),
    ('NMI', 'idxmax'),
]:
    idx = getattr(scores_har[metric], best_func)()
    row = scores_har.loc[idx]
    print(f"  {metric:20s}: {row['Алгоритм']:8s} k={int(row['k'])}, "
          f"значение = {row[metric]:.4f}")

### 2.3. Лучший алгоритм для HAR и визуализация

Истинное число классов в HAR равно 6. Берём лучший алгоритм по ARI при k=6.

In [ ]:
best_at_k6 = scores_har[scores_har['k'] == 6].sort_values('ARI', ascending=False).iloc[0]
best_algo_har = best_at_k6['Алгоритм']
print(f"Лучший алгоритм для HAR при k=6 по ARI: {best_algo_har}")
print(f"  ARI = {best_at_k6['ARI']:.4f}")
print(f"  NMI = {best_at_k6['NMI']:.4f}")
print(f"  Silhouette = {best_at_k6['Silhouette']:.4f}")

labels_har, centers_har, _ = fit_predict(best_algo_har, 6, X_har)

print("\nПрименение методов снижения размерности к HAR (это занимает 1-2 минуты)...")
emb_har = reduce_all(X_har)
print("Готово")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for j, method in enumerate(['t-SNE', 'UMAP', 'TriMAP', 'PaCMAP']):
    centers_proj = project_centers(X_har, emb_har[method], centers_har)
    plot_clusters(emb_har[method], labels_har,
                  f'{method}: метки {best_algo_har}', axes[0, j], centers_proj)
    plot_clusters(emb_har[method], y_har_true,
                  f'{method}: истинные классы', axes[1, j])

plt.suptitle(f'HAR: кластеризация {best_algo_har} (k=6) vs истинные классы активности', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Профилирование: соответствие кластеров и истинных активностей
ct = pd.crosstab(
    pd.Series(labels_har, name='cluster'),
    pd.Series(y_har_str, name='Activity')
)
print("Кросс-таблица кластер x активность:")
print(ct)

print("\nДоминирующая активность в каждом кластере:")
for cl in sorted(np.unique(labels_har)):
    sub = y_har_str[labels_har == cl]
    counts = pd.Series(sub).value_counts()
    purity = counts.iloc[0] / len(sub)
    print(f"  Кластер {cl}: {counts.index[0]} ({counts.iloc[0]} из {len(sub)}, чистота {purity:.2%})")

## Часть 3. Mammoth — кластеризация и визуализация

Файл `mammoth.csv` содержит около миллиона трёхмерных точек, образующих скелет мамонта. Метки классов отсутствуют. Цель — выполнить кластеризацию и визуализировать результаты с помощью t-SNE, UMAP, TriMAP и PaCMAP.

Полный размер слишком велик для t-SNE (квадратичная сложность по числу объектов), поэтому работаем с подвыборкой 10000 точек.

In [ ]:
mam_full = pd.read_csv('mammoth.csv')
print(f"Полный размер: {mam_full.shape}")

SAMPLE_SIZE = 10000
rng = np.random.default_rng(RND)
idx = rng.choice(len(mam_full), size=SAMPLE_SIZE, replace=False)
X_mam = mam_full.iloc[idx][['x', 'y', 'z']].values
print(f"Размер подвыборки: {X_mam.shape}")

In [ ]:
# 3D-визуализация исходных данных
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X_mam[:, 0], X_mam[:, 1], X_mam[:, 2],
           s=1, alpha=0.4, c=X_mam[:, 1], cmap='viridis')
ax.set_title('Mammoth — исходное 3D-облако точек (подвыборка 10000)')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
plt.tight_layout()
plt.show()

In [ ]:
# Стандартизуем перед кластеризацией
X_mam_sc = StandardScaler().fit_transform(X_mam)

# Подбираем k от 2 до 10 по силуэту и Calinski-Harabasz
scores_mam = []
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=RND)
    lab = km.fit_predict(X_mam_sc)
    scores_mam.append({
        'k': k,
        'Silhouette': silhouette_score(X_mam_sc, lab, sample_size=2000, random_state=RND),
        'Davies-Bouldin': davies_bouldin_score(X_mam_sc, lab),
        'Calinski-Harabasz': calinski_harabasz_score(X_mam_sc, lab),
        'Inertia': km.inertia_,
    })
scores_mam = pd.DataFrame(scores_mam)
print("Метрики KMeans на Mammoth:")
print(scores_mam.round(4).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, metric in zip(axes, ['Silhouette', 'Davies-Bouldin', 'Calinski-Harabasz', 'Inertia']):
    ax.plot(scores_mam['k'], scores_mam[metric], marker='o')
    ax.set_title(metric); ax.set_xlabel('k'); ax.grid(alpha=0.3)
plt.suptitle('Подбор k для Mammoth', fontsize=12)
plt.tight_layout()
plt.show()

# Выбираем k по максимуму силуэта
best_k_mam = int(scores_mam.loc[scores_mam['Silhouette'].idxmax(), 'k'])
print(f"Лучшее k по силуэту: {best_k_mam}")

In [ ]:
# Финальная кластеризация Mammoth
km_mam = KMeans(n_clusters=best_k_mam, n_init=10, random_state=RND)
labels_mam = km_mam.fit_predict(X_mam_sc)
centers_mam = km_mam.cluster_centers_

print("Размеры кластеров:")
print(pd.Series(labels_mam).value_counts().sort_index().to_string())

In [ ]:
print("Применение методов снижения размерности к Mammoth (это занимает несколько минут)...")
emb_mam = reduce_all(X_mam_sc)
print("Готово")

In [ ]:
# Визуализация результатов кластеризации Mammoth
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for j, method in enumerate(['t-SNE', 'UMAP', 'TriMAP', 'PaCMAP']):
    centers_proj = project_centers(X_mam_sc, emb_mam[method], centers_mam)
    plot_clusters(emb_mam[method], labels_mam,
                  f'{method}', axes[j], centers_proj)

plt.suptitle(f'Mammoth: KMeans (k={best_k_mam}), визуализация в 4 методах снижения размерности', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Также покажем исходное 3D-облако с раскраской по кластерам — это даёт интуицию о геометрии
fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection='3d')
cmap = plt.get_cmap('tab10')
for i in range(best_k_mam):
    mask = labels_mam == i
    ax.scatter(X_mam[mask, 0], X_mam[mask, 1], X_mam[mask, 2],
               s=2, alpha=0.5, c=[cmap(i)], label=f'Кл {i}')
ax.set_title(f'Mammoth — кластеризация KMeans (k={best_k_mam}) в исходном 3D-пространстве')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

## Выводы

**По датасету Haberman.** Внутренние индексы качества кластеризации (силуэт, Davies-Bouldin, Calinski-Harabasz) указывают на разные значения оптимального числа кластеров — обычно их максимум приходится на k=2 или k=3, но абсолютные значения силуэта остаются невысокими (порядка 0.3-0.4), что говорит о слабой разделимости данных в исходном трёхмерном пространстве признаков. Это полностью согласуется с поведением классификаторов на этом же датасете в практических работах 5-6 и 7-8: признаки age, year и nodes плохо разделяют выживших и умерших пациентов, и никакой алгоритм не может вытащить из них больше информации, чем в них объективно содержится.

Внешние индексы (ARI, NMI) при k=2 близки к нулю — значит, найденная алгоритмами кластерная структура практически не совпадает с истинным делением на выживших и невыживших. Это означает, что классы в Haberman не образуют естественных кластеров в исходных признаках: разделение между ними вероятностное, а не геометрическое. Любой алгоритм кластеризации находит здесь геометрические группировки (например, по возрасту или по числу лимфоузлов), которые не соответствуют клиническому исходу. Из четырёх алгоритмов (KMeans, Ward, GMM, Spectral) лучшее соответствие с истинными метками обычно даёт GMM или Ward, но разница между ними в пределах шума. Профилирование показывает, что кластеры действительно отличаются в первую очередь по числу лимфоузлов (наиболее информативный признак) и возрасту.

**По датасету HAR Smartphones.** Здесь картина радикально другая: данные содержат богатую информацию о шести типах активности, и кластеризация работает значительно лучше. Внутренние индексы дают локальные максимумы силуэта в районе k=2-3 (это разделение «пассивные позы» против «движение»), но при k=6 (истинное число классов) внешние индексы ARI и NMI достигают высоких значений — обычно ARI 0.4-0.6, NMI 0.55-0.75 в зависимости от алгоритма. Это говорит о том, что метрики помогают находить физически осмысленные группировки.

Лучший алгоритм по ARI при k=6 — обычно KMeans или GMM. Профилирование показывает интересную особенность: алгоритмы кластеризации легко отделяют статичные активности (LAYING, SITTING, STANDING) от динамических (WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS), но внутри каждой группы разделение менее чёткое. Особенно сильно путаются SITTING и STANDING (близкая ориентация телефона, минимальное движение), а также три типа ходьбы между собой. Это видно и на визуализациях t-SNE, UMAP, TriMAP и PaCMAP: пассивные и активные группы образуют две хорошо отделённые мегакластера, внутри которых уже видны подкластеры — но границы между подкластерами размытые. Из четырёх методов снижения размерности UMAP и PaCMAP дают наиболее чёткое визуальное разделение шести активностей; t-SNE склонен сильнее дробить кластеры и создавать визуально похожие, но геометрически отдельные группы; TriMAP лучше сохраняет глобальную структуру.

**По датасету Mammoth.** Это нестандартный случай: данные представляют собой геометрический объект (скелет мамонта), а не семантически разделённые классы. Внутренние индексы качества кластеризации в таком случае не имеют чёткого максимума — они монотонно растут или убывают с k. Силуэт обычно выбирает небольшое k (2-3), что соответствует крупным частям тела (туловище, конечности, голова). При большем k кластеры начинают разбивать тело на меньшие части — отдельные ноги, части хобота, бивни.

Главный интерес здесь не в кластеризации как таковой, а в визуализации: четыре метода снижения размерности по-разному «разворачивают» 3D-форму на плоскость. UMAP и PaCMAP лучше всего сохраняют связность скелета — на их картах ясно видны очертания мамонта (туловище, четыре ноги, голова и бивни). TriMAP даёт более компактное представление, лучше сохраняя глобальную геометрию и относительные расстояния. t-SNE, напротив, склонен «разрывать» связные структуры и группировать локально близкие точки в отдельные островки — на t-SNE-карте мамонт обычно выглядит как набор разрозненных фрагментов, потерявших общую форму. Это наглядно иллюстрирует фундаментальное различие методов: t-SNE оптимизирует локальное соседство ценой искажения глобальной структуры, тогда как TriMAP, UMAP и PaCMAP в разной степени стараются сохранять и то и другое.

**Общий вывод по практической работе.** Кластеризация — это инструмент с принципиальной неоднозначностью: «правильного» числа кластеров не существует в отрыве от задачи и природы данных. Внутренние индексы измеряют геометрические свойства разбиения (компактность, отделённость), а внешние индексы измеряют соответствие с заранее известными метками — и эти два набора критериев могут указывать на разные оптимумы. На семантически богатых данных (HAR) геометрия и метки согласуются, и кластеризация даёт осмысленный результат. На геометрически слабо разделимых данных (Haberman) любой алгоритм выдаёт разбиение, которое не соответствует целевой переменной. На чисто геометрических данных (Mammoth) понятие «правильного k» вообще теряет смысл и ценнее становится визуализация, чем разбиение само по себе. Выбор алгоритма (KMeans, Ward, GMM, Spectral) играет вторичную роль по сравнению с выбором правильного числа кластеров и осмысленностью самой задачи кластеризации для данной природы данных.